In [1]:
import cv2
import numpy as np
import pandas as pd
from scipy import signal, stats
from scipy.ndimage import gaussian_filter1d
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib import gridspec

# **Code for Data extraction from image**

In [2]:
class ECGWaveformExtractor:
    """Complete pipeline for extracting waveform data from 12-lead ECG images"""
    
    def __init__(self, sampling_rate=500):
        self.sampling_rate = sampling_rate
        self.lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 
                          'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

    def preprocess_image(self, image_path):
        img = cv2.imread(image_path)
        left_pixels = 37
        right_pixels = 20
        if img is None:
            raise ValueError(f"Could not load image: {image_path}")
        if img.ndim == 2:
            cropped = img[:, left_pixels:img.shape[1]-right_pixels]
        else:
            cropped = img[:, left_pixels:img.shape[1]-right_pixels, :]
        
        gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
        original_gray = gray.copy()
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        
        return {'original': img, 'grayscale': original_gray, 'binary': binary}
    
    def remove_grid_lines(self, binary_img, kernel_size=3):
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
        vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25))
        
        horizontal_lines = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, horizontal_kernel)
        vertical_lines = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, vertical_kernel)
        
        grid = cv2.add(horizontal_lines, vertical_lines)
        cleaned = cv2.subtract(binary_img, grid)
        
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
        
        return cleaned
    
    def detect_leads_and_rhythm(self, cleaned_img):
        height, width = cleaned_img.shape
        horizontal_projection = np.sum(cleaned_img, axis=1)
        smooth_projection = gaussian_filter1d(horizontal_projection, sigma=10)
        peaks, _ = signal.find_peaks(smooth_projection, distance=height//6, prominence=width*5)
        
        lead_regions = []
        rhythm_region = None
        
        if len(peaks) >= 4:
            row_boundaries = []
            for i in range(len(peaks)):
                if i == 0:
                    y_start = 0
                else:
                    y_start = (peaks[i-1] + peaks[i]) // 2
                
                if i == len(peaks) - 1:
                    y_end = height
                else:
                    y_end = (peaks[i] + peaks[i+1]) // 2
                
                row_boundaries.append((y_start, y_end))
            
            for i in range(3):
                y_start, y_end = row_boundaries[i]
                col_width = width // 4
                
                for j in range(4):
                    x_start = j * col_width
                    x_end = (j + 1) * col_width if j < 3 else width
                    trim_amount = int((x_end - x_start) * 0.08)
                    x_start_trimmed = x_start + trim_amount
                    lead_regions.append((y_start, y_end, x_start_trimmed, x_end))
            
            y_start, y_end = row_boundaries[3]
            trim_amount = int(width * 0.05)
            rhythm_region = (y_start, y_end, trim_amount, width)
        else:
            row_height = height // 4
            
            for i in range(3):
                y_start = i * row_height
                y_end = (i + 1) * row_height
                col_width = width // 4
                
                for j in range(4):
                    x_start = j * col_width
                    x_end = (j + 1) * col_width if j < 3 else width
                    trim_amount = int((x_end - x_start) * 0.08)
                    x_start_trimmed = x_start + trim_amount
                    lead_regions.append((y_start, y_end, x_start_trimmed, x_end))
            
            y_start = 3 * row_height
            y_end = height
            trim_amount = int(width * 0.05)
            rhythm_region = (y_start, y_end, trim_amount, width)
        
        return lead_regions, rhythm_region
    
    def extract_signal_from_region(self, cleaned_img, region):
        y_start, y_end, x_start, x_end = region
        lead_img = cleaned_img[y_start:y_end, x_start:x_end]
        
        signal_data = []
        for col in range(lead_img.shape[1]):
            column = lead_img[:, col]
            white_pixels = np.where(column > 0)[0]
            
            if len(white_pixels) > 0:
                signal_point = np.median(white_pixels)
            else:
                signal_point = np.nan
            
            signal_data.append(signal_point)
        
        signal_data = np.array(signal_data)
        nans = np.isnan(signal_data)
        if np.any(nans):
            x = np.arange(len(signal_data))
            signal_data[nans] = np.interp(x[nans], x[~nans], signal_data[~nans])
        
        signal_data = lead_img.shape[0] - signal_data
        signal_data = signal_data - np.mean(signal_data)
        
        return signal_data
    
    def process_ecg_image(self, image_path, visualize=False):
        preprocessed = self.preprocess_image(image_path)
        cleaned = self.remove_grid_lines(preprocessed['binary'])
        lead_regions, rhythm_region = self.detect_leads_and_rhythm(cleaned)
        
        twelve_leads = {}
        for i, region in enumerate(lead_regions):
            lead_name = self.lead_names[i] if i < len(self.lead_names) else f"Lead_{i+1}"
            raw_signal = self.extract_signal_from_region(cleaned, region)
            twelve_leads[lead_name] = {'signal': raw_signal, 'region': region}
        
        extended_lead_II = None
        if rhythm_region is not None:
            raw_signal = self.extract_signal_from_region(cleaned, rhythm_region)
            extended_lead_II = {'signal': raw_signal, 'region': rhythm_region}
        
        result = {
            '12_leads': twelve_leads,
            'extended_lead_II': extended_lead_II
        }
        
        return result

# **Code for Feature Extraction from ECG Columns**

In [3]:
def calculate_mav(sig):
    return np.mean(np.abs(sig))

def calculate_rms(sig):
    return np.sqrt(np.mean(sig**2))

def calculate_zc(sig, threshold=0):
    sign_changes = np.diff(np.sign(sig))
    return np.sum(np.abs(sign_changes) > threshold)

def calculate_ssc(sig, threshold=0):
    diff_signal = np.diff(sig)
    sign_changes = np.diff(np.sign(diff_signal))
    return np.sum(np.abs(sign_changes) > threshold)

def calculate_var(sig):
    return np.var(sig)

def calculate_dasdv(sig):
    diff_signal = np.diff(sig)
    return np.sqrt(np.mean(diff_signal**2))

def calculate_aac(sig):
    return np.mean(np.abs(np.diff(sig)))

def calculate_skewness(sig):
    return stats.skew(sig)

def calculate_kurtosis(sig):
    return stats.kurtosis(sig)

def extract_features_from_signal(sig):
    features = {
        'MAV': calculate_mav(sig),
        'RMS': calculate_rms(sig),
        'ZC': calculate_zc(sig),
        'SSC': calculate_ssc(sig),
        'VAR': calculate_var(sig),
        'DASDV': calculate_dasdv(sig),
        'AAC': calculate_aac(sig),
        'Skew': calculate_skewness(sig),
        'Kurt': calculate_kurtosis(sig)
    }
    return features

# **Final Pipeline for data Creation**

In [ ]:
def process_complete_pipeline(root_folder, output_folder, visualize_first=False):
    """
    Complete end-to-end pipeline:
    1. Extract signals from ECG images
    2. Save as NPZ files
    3. Convert to CSV files
    4. Extract features
    5. Create final summary CSV with everything
    
    Args:
        root_folder (str): Path to folder with class subfolders containing ECG images
        output_folder (str): Path to save all outputs
        visualize_first (bool): Visualize first image processing
    
    Returns:
        tuple: (summary_12leads, summary_rhythm, final_combined_csv)
    """
    
    root_path = Path(root_folder)
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Create folder structure
    npz_12leads = output_path / "npz_data" / "12_leads_data"
    npz_rhythm = output_path / "npz_data" / "extended_lead_II_data"
    csv_12leads = output_path / "csv_data" / "12_leads_csv"
    csv_rhythm = output_path / "csv_data" / "extended_lead_II_csv"
    
    npz_12leads.mkdir(parents=True, exist_ok=True)
    npz_rhythm.mkdir(parents=True, exist_ok=True)
    csv_12leads.mkdir(parents=True, exist_ok=True)
    csv_rhythm.mkdir(parents=True, exist_ok=True)
    
    # Initialize extractor
    extractor = ECGWaveformExtractor(sampling_rate=500)
    
    # Get all class folders
    class_folders = [f for f in root_path.iterdir() if f.is_dir()]
    
    all_records_12leads = []
    all_records_rhythm = []
    
    print("="*80)
    print("STARTING COMPLETE ECG PROCESSING PIPELINE")
    print("="*80)
    
    for class_idx, class_folder in enumerate(class_folders):
        class_name = class_folder.name
        print(f"\n{'='*80}")
        print(f"Processing Class: {class_name}")
        print(f"{'='*80}")
        
        # Create class subfolders
        class_npz_12leads = npz_12leads / class_name
        class_npz_rhythm = npz_rhythm / class_name
        class_csv_12leads = csv_12leads / class_name
        class_csv_rhythm = csv_rhythm / class_name
        
        class_npz_12leads.mkdir(parents=True, exist_ok=True)
        class_npz_rhythm.mkdir(parents=True, exist_ok=True)
        class_csv_12leads.mkdir(parents=True, exist_ok=True)
        class_csv_rhythm.mkdir(parents=True, exist_ok=True)
        
        # Get all images
        image_files = list(class_folder.glob('*.png')) + \
                     list(class_folder.glob('*.jpg')) + \
                     list(class_folder.glob('*.jpeg')) + \
                     list(class_folder.glob('*.bmp'))
        
        print(f"Found {len(image_files)} images")
        
        for idx, img_path in enumerate(tqdm(image_files, desc=f"Processing {class_name}")):
            try:
                # STEP 1: Extract signals from image
                visualize = visualize_first and idx == 0 and class_idx == 0
                result = extractor.process_ecg_image(str(img_path), visualize=visualize)
                
                twelve_leads = result['12_leads']
                extended_lead_II = result['extended_lead_II']
                
                # ========== PROCESS 12-LEAD DATA ==========
                # Save NPZ
                npz_file_12leads = class_npz_12leads / f"{img_path.stem}_12leads.npz"
                save_data_12leads = {
                    'lead_names': extractor.lead_names,
                    'class': class_name,
                    'original_image': str(img_path)
                }
                for lead_name, data in twelve_leads.items():
                    save_data_12leads[f'{lead_name}_signal'] = data['signal']
                
                np.savez_compressed(npz_file_12leads, **save_data_12leads)
                
                # Convert to CSV
                csv_file_12leads = class_csv_12leads / f"{img_path.stem}.csv"
                df_data = {}
                for lead_name, data in twelve_leads.items():
                    df_data[lead_name] = data['signal']
                df = pd.DataFrame(df_data)
                df.to_csv(csv_file_12leads, index=False)
                
                # Extract features
                record_12leads = {
                    'original_image': str(img_path),
                    'npz_path': str(npz_file_12leads),
                    'csv_path': str(csv_file_12leads),
                    'data_type': '12_leads',
                    'class': class_name,
                    'num_samples': len(df),
                    'num_leads': len(df.columns),
                    'status': 'success'
                }
                
                # Extract features from each lead
                lead_signal_names = ['I_signal', 'II_signal', 'III_signal', 
                                   'aVR_signal', 'aVL_signal', 'aVF_signal',
                                   'V1_signal', 'V2_signal', 'V3_signal', 
                                   'V4_signal', 'V5_signal', 'V6_signal']
                
                npz_data = np.load(npz_file_12leads, allow_pickle=True)
                for col_idx, signal_name in enumerate(lead_signal_names, 1):
                    if signal_name in npz_data.files:
                        sig = npz_data[signal_name]
                        features = extract_features_from_signal(sig)
                        for feature_name, feature_value in features.items():
                            record_12leads[f'Col{col_idx}_{feature_name}'] = feature_value
                
                all_records_12leads.append(record_12leads)
                
                # ========== PROCESS EXTENDED LEAD II DATA ==========
                if extended_lead_II is not None:
                    # Save NPZ
                    npz_file_rhythm = class_npz_rhythm / f"{img_path.stem}_extended_leadII.npz"
                    save_data_rhythm = {
                        'class': class_name,
                        'original_image': str(img_path),
                        'signal': extended_lead_II['signal']
                    }
                    np.savez_compressed(npz_file_rhythm, **save_data_rhythm)
                    
                    # Convert to CSV
                    csv_file_rhythm = class_csv_rhythm / f"{img_path.stem}.csv"
                    df_rhythm = pd.DataFrame({
                        'Extended_Lead_II': extended_lead_II['signal']
                    })
                    df_rhythm.to_csv(csv_file_rhythm, index=False)
                    
                    # Extract features
                    record_rhythm = {
                        'original_image': str(img_path),
                        'npz_path': str(npz_file_rhythm),
                        'csv_path': str(csv_file_rhythm),
                        'data_type': 'extended_lead_II',
                        'class': class_name,
                        'num_samples': len(df_rhythm),
                        'status': 'success'
                    }
                    
                    sig = extended_lead_II['signal']
                    features = extract_features_from_signal(sig)
                    for feature_name, feature_value in features.items():
                        record_rhythm[feature_name] = feature_value
                    
                    all_records_rhythm.append(record_rhythm)
                
            except Exception as e:
                print(f"\n✗ Error processing {img_path.name}: {str(e)}")
                all_records_12leads.append({
                    'original_image': str(img_path),
                    'npz_path': None,
                    'csv_path': None,
                    'data_type': '12_leads',
                    'class': class_name,
                    'status': 'failed',
                    'error': str(e)
                })
    
    # Create summary DataFrames
    print(f"\n{'='*80}")
    print("CREATING SUMMARY FILES")
    print(f"{'='*80}")
    
    summary_12leads = pd.DataFrame(all_records_12leads)
    summary_rhythm = pd.DataFrame(all_records_rhythm)
    
    # Reorder columns for 12-leads
    if len(summary_12leads) > 0:
        metadata_cols = ['original_image', 'npz_path', 'csv_path', 'data_type', 
                        'num_samples', 'num_leads', 'status']
        feature_cols = [col for col in summary_12leads.columns 
                       if col not in metadata_cols and col not in ['class', 'error']]
        final_cols = metadata_cols + feature_cols + ['class']
        if 'error' in summary_12leads.columns:
            final_cols.append('error')
        final_cols = [col for col in final_cols if col in summary_12leads.columns]
        summary_12leads = summary_12leads[final_cols]
        
        summary_12leads_path = output_path / 'summary_12_leads_complete.csv'
        summary_12leads.to_csv(summary_12leads_path, index=False)
        print(f"✓ 12-leads summary saved: {summary_12leads_path}")
    
    # Reorder columns for rhythm
    if len(summary_rhythm) > 0:
        metadata_cols = ['original_image', 'npz_path', 'csv_path', 'data_type', 
                        'num_samples', 'status']
        feature_cols = [col for col in summary_rhythm.columns 
                       if col not in metadata_cols and col not in ['class', 'error']]
        final_cols = metadata_cols + feature_cols + ['class']
        if 'error' in summary_rhythm.columns:
            final_cols.append('error')
        final_cols = [col for col in final_cols if col in summary_rhythm.columns]
        summary_rhythm = summary_rhythm[final_cols]
        
        summary_rhythm_path = output_path / 'summary_extended_lead_II_complete.csv'
        summary_rhythm.to_csv(summary_rhythm_path, index=False)
        print(f"✓ Extended Lead II summary saved: {summary_rhythm_path}")
    
    # Create combined final CSV
    combined = None
    if len(summary_12leads) > 0 and len(summary_rhythm) > 0:
        combined = pd.concat([summary_12leads, summary_rhythm], ignore_index=True)
    elif len(summary_12leads) > 0:
        combined = summary_12leads
    elif len(summary_rhythm) > 0:
        combined = summary_rhythm
    
    if combined is not None:
        combined_path = output_path / 'FINAL_COMBINED_SUMMARY.csv'
        combined.to_csv(combined_path, index=False)
        print(f"✓ Final combined summary saved: {combined_path}")
    
    # Print final statistics
    print(f"\n{'='*80}")
    print("PIPELINE COMPLETE!")
    print(f"{'='*80}")
    
    if len(summary_12leads) > 0:
        print(f"\n12-LEAD DATA:")
        print(f"  Total: {len(summary_12leads)}")
        print(f"  Success: {len(summary_12leads[summary_12leads['status'] == 'success'])}")
        print(f"  Failed: {len(summary_12leads[summary_12leads['status'] == 'failed'])}")
    
    if len(summary_rhythm) > 0:
        print(f"\nEXTENDED LEAD II DATA:")
        print(f"  Total: {len(summary_rhythm)}")
        print(f"  Success: {len(summary_rhythm[summary_rhythm['status'] == 'success'])}")
        print(f"  Failed: {len(summary_rhythm[summary_rhythm['status'] == 'failed'])}")
    
    if combined is not None:
        print(f"\nCOMBINED DATA:")
        print(f"  Total records: {len(combined)}")
        print(f"\nClass Distribution:")
        for class_name, count in combined[combined['status'] == 'success'].groupby('class').size().items():
            print(f"  {class_name}: {count} samples")
    
    print(f"\n{'='*80}")
    print("OUTPUT STRUCTURE:")
    print(f"{'='*80}")
    print(f"""
{output_folder}/
├── npz_data/
│   ├── 12_leads_data/
│   │   └── [class folders with .npz files]
│   └── extended_lead_II_data/
│       └── [class folders with .npz files]
├── csv_data/
│   ├── 12_leads_csv/
│   │   └── [class folders with .csv signal files]
│   └── extended_lead_II_csv/
│       └── [class folders with .csv signal files]
├── summary_12_leads_complete.csv (all features + paths + labels)
├── summary_extended_lead_II_complete.csv (all features + paths + labels)
└── FINAL_COMBINED_SUMMARY.csv (EVERYTHING in one file!)
    """)
    
    return summary_12leads, summary_rhythm, combined


In [5]:
if __name__ == "__main__":
    # Set your paths
    root_folder = "./ECG Dataset"  # Folder with class subfolders containing images
    output_folder = "./final_check_output"  # Output folder for everything
    
    # Run complete pipeline
    summary_12leads, summary_rhythm, final_combined = process_complete_pipeline(
        root_folder=root_folder,
        output_folder=output_folder,
        visualize_first=True  # Visualize first image
    )
    
    print("\n" + "="*80)
    print("✓ ALL DONE! Check FINAL_COMBINED_SUMMARY.csv for complete data")
    print("="*80)

STARTING COMPLETE ECG PROCESSING PIPELINE

Processing Class: Abnormal heartbeat
Found 241 images


Processing Abnormal heartbeat: 100%|██████████| 241/241 [00:23<00:00, 10.07it/s]



Processing Class: History of MI
Found 171 images


Processing History of MI: 100%|██████████| 171/171 [00:19<00:00,  8.60it/s]



Processing Class: Normal Person
Found 295 images


Processing Normal Person: 100%|██████████| 295/295 [00:32<00:00,  9.16it/s]



CREATING SUMMARY FILES
✓ 12-leads summary saved: final_check_output\summary_12_leads_complete.csv
✓ Extended Lead II summary saved: final_check_output\summary_extended_lead_II_complete.csv
✓ Final combined summary saved: final_check_output\FINAL_COMBINED_SUMMARY.csv

PIPELINE COMPLETE!

12-LEAD DATA:
  Total: 707
  Success: 707
  Failed: 0

EXTENDED LEAD II DATA:
  Total: 707
  Success: 707
  Failed: 0

COMBINED DATA:
  Total records: 1414

Class Distribution:
  Abnormal heartbeat: 482 samples
  History of MI: 342 samples
  Normal Person: 590 samples

OUTPUT STRUCTURE:

./final_check_output/
├── npz_data/
│   ├── 12_leads_data/
│   │   └── [class folders with .npz files]
│   └── extended_lead_II_data/
│       └── [class folders with .npz files]
├── csv_data/
│   ├── 12_leads_csv/
│   │   └── [class folders with .csv signal files]
│   └── extended_lead_II_csv/
│       └── [class folders with .csv signal files]
├── summary_12_leads_complete.csv (all features + paths + labels)
├── summar

# **Data Cleaning and Preparation**

In [6]:
df_12_lead = pd.read_csv('final_check_output/summary_12_leads_complete.csv')
df_rhythm = pd.read_csv('final_check_output/summary_extended_lead_II_complete.csv')

In [7]:
def label_encode(x):
    if x=="Normal Person":
        return 0
    elif x=="Abnormal heartbeat":
        return 1
    else:
        return 2
df_12_lead['class_encoded'] = df_12_lead['class'].apply(label_encode)
df_rhythm['class_encoded'] = df_rhythm['class'].apply(label_encode)

In [8]:
df_12_lead = df_12_lead.drop(columns=['original_image', 'npz_path', 'csv_path', 'data_type', 'status', 'class'])
df_rhythm = df_rhythm.drop(columns=['original_image', 'npz_path', 'csv_path', 'data_type', 'status', 'class'])

In [9]:
df_12_lead.to_csv('final_final_12_lead_data.csv')
df_rhythm.to_csv('final_final_rhythm_data.csv')

In [52]:
import numpy as np
import pandas as pd

In [12]:
# Step 1: Separate features and labels
X = df_12_lead.drop('class_encoded', axis=1)
y = df_12_lead['class_encoded']

print("Original class distribution:")
print(y.value_counts().sort_index())
print(f"Total samples: {len(y)}")


Original class distribution:
class_encoded
0    295
1    241
2    171
Name: count, dtype: int64
Total samples: 707


# **Balancing the Dataset**

In [13]:
def balance_dataset(X, y):
    """Balance dataset by undersampling majority classes"""
    df_combined = pd.concat([X, y], axis=1)
    min_class_size = y.value_counts().min()
    print(f"\nBalancing to {min_class_size} samples per class")
    
    balanced_dfs = []
    for class_label in y.unique():
        class_df = df_combined[df_combined['class_encoded'] == class_label]
        class_df_balanced = resample(class_df, n_samples=min_class_size, 
                                     random_state=42, replace=False)
        balanced_dfs.append(class_df_balanced)
    
    df_balanced = pd.concat(balanced_dfs, axis=0)
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    
    X_balanced = df_balanced.drop('class_encoded', axis=1)
    y_balanced = df_balanced['class_encoded']
    
    return X_balanced, y_balanced

In [14]:
X_balanced, y_balanced = balance_dataset(X, y)
print("\nBalanced class distribution:")
print(y_balanced.value_counts().sort_index())


Balancing to 171 samples per class

Balanced class distribution:
class_encoded
0    171
1    171
2    171
Name: count, dtype: int64


# **Model Training**

## **ECG 12 Lead Data**

In [41]:
# combined_pipeline.py
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

# sklearn imports
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

# gradient boosting / other libraries
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# keras for DNN
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# ---------------------------
# 1) model factory
# ---------------------------
def get_models(n_classes=None):
    """
    Returns a dict of model name -> model object.
    If n_classes is provided, GMM's n_components will be set accordingly.
    """
    models = {
        'SVM (Linear)': SVC(kernel='linear', probability=False, random_state=42),
        'SVM (RBF Kernel)': SVC(kernel='rbf', probability=False, random_state=42, gamma='scale'),
        'SVM (Poly Kernel)': SVC(kernel='poly', degree=3, probability=False, random_state=42, gamma='scale'),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False),
        'MLP': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42, early_stopping=True),
        'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
        'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42),
        'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
        # GMM will be handled specially during training if present
        'GMM': GaussianMixture(n_components=(n_classes if n_classes is not None else 3),
                               covariance_type='full',
                               random_state=42),
        'LightGBM': LGBMClassifier(random_state=42)
    }
    return models

# ---------------------------
# 2) DNN builder (Keras)
# ---------------------------
def build_dnn(input_dim, num_classes, lr=1e-3):
    model = Sequential([
        Dense(256, activation='relu', input_dim=input_dim),
        BatchNormalization(),
        Dropout(0.4),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ---------------------------
# 3) Helper: GMM supervised wrapper
# ---------------------------
def gmm_supervised_predict(gmm, X_train_scaled, y_train, X_val_scaled):
    """
    Fit GMM on X_train_scaled, map clusters to labels using majority vote on train,
    then predict labels for X_val_scaled.
    """
    # Fit GMM on training features
    gmm.fit(X_train_scaled)
    train_clusters = gmm.predict(X_train_scaled)

    # Map cluster -> most common true label in that cluster
    mapping = {}
    for cluster_id in np.unique(train_clusters):
        idxs = np.where(train_clusters == cluster_id)[0]
        labels_in_cluster = y_train.iloc[idxs].values if hasattr(y_train, "iloc") else y_train[ idxs ]
        most_common = Counter(labels_in_cluster).most_common(1)[0][0]
        mapping[cluster_id] = most_common

    # Predict clusters for val set then map
    val_clusters = gmm.predict(X_val_scaled)
    y_pred = np.array([mapping.get(c, -1) for c in val_clusters]).flatten()
    return y_pred

# ---------------------------
# 4) Main training + CV routine
# ---------------------------
def run_all_models_with_kfold(X, y, n_splits=10, random_state=42, include_dnn=True, dnn_epochs=50, dnn_batch=32):
    """
    Trains all models from get_models() + optional DNN using Stratified K-Fold CV.
    Returns:
      - all_results: {model_name: [ {fold, accuracy, y_val, y_pred}, ... ] }
      - model_summaries: {model_name: {accuracy, precision, recall, f1, support, confusion_matrix, min, max, std} }
    """
    # Label encode y if it's string labels
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    classes = le.classes_
    n_classes = len(classes)

    models = get_models(n_classes=n_classes)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    all_results = defaultdict(list)

    fold_idx = 0
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_enc), 1):
        print(f"\n=== Fold {fold}/{n_splits} ===")
        fold_idx = fold

        X_train = X.iloc[train_idx] if isinstance(X, pd.DataFrame) else X[train_idx]
        X_val = X.iloc[val_idx] if isinstance(X, pd.DataFrame) else X[val_idx]
        y_train = pd.Series(y_enc[train_idx])
        y_val = pd.Series(y_enc[val_idx])

        # Scale
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        # Train each sklearn-style model
        for name, model in models.items():
            print(f" Training {name} ...", end=' ')
            # Skip DNN here; handle separately
            if name == 'GMM':
                # GMM special supervised wrapper
                try:
                    y_pred = gmm_supervised_predict(model, X_train_scaled, y_train, X_val_scaled)
                    acc = accuracy_score(y_val, y_pred)
                    print(f"acc={acc:.4f}")
                except Exception as e:
                    print(f" GMM failed: {e}")
                    y_pred = np.full(len(y_val), -1)
                    acc = 0.0
            else:
                try:
                    # CatBoost supports raw numpy/pandas; for consistency, pass arrays
                    model.fit(X_train_scaled, y_train)
                    y_pred = model.predict(X_val_scaled)
                    y_pred = np.array(y_pred).flatten()
                    # some models return floats; ensure ints
                    if y_pred.dtype.kind == 'f':
                        y_pred = np.rint(y_pred).astype(int)
                    acc = accuracy_score(y_val, y_pred)
                    print(f"acc={acc:.4f}")
                except Exception as e:
                    # in case of unexpected failure
                    print(f" failed: {e}")
                    y_pred = np.full(len(y_val), -1)
                    acc = 0.0

            all_results[name].append({
                'fold': fold,
                'accuracy': acc,
                'y_val': y_val.values,
                'y_pred': np.array(y_pred)
            })

        # Train DNN separately if requested
        if include_dnn:
            dnn_name = 'DNN'
            print(f" Training {dnn_name} ...", end=' ')
            try:
                tf.keras.backend.clear_session()
                dnn = build_dnn(input_dim=X_train_scaled.shape[1], num_classes=n_classes)
                es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0)
                dnn.fit(X_train_scaled, y_train.values,
                        validation_data=(X_val_scaled, y_val.values),
                        epochs=dnn_epochs,
                        batch_size=dnn_batch,
                        verbose=0,
                        callbacks=[es])
                y_prob = dnn.predict(X_val_scaled)
                y_pred = np.argmax(y_prob, axis=1).flatten()

                acc = accuracy_score(y_val, y_pred)
                print(f"acc={acc:.4f}")
            except Exception as e:
                print(f" failed: {e}")
                y_pred = np.full(len(y_val), -1)
                acc = 0.0

            all_results[dnn_name].append({
                'fold': fold,
                'accuracy': acc,
                'y_val': y_val.values,
                'y_pred': np.array(y_pred)
            })

    # ---------------------------
    # Aggregate per-model summaries
    # ---------------------------
    model_summaries = {}
    for model_name, results in all_results.items():
        # concatenated arrays across folds
        all_y_val = np.concatenate([np.array(r['y_val']).flatten() for r in results])
        all_y_pred = np.concatenate([np.array(r['y_pred']).flatten() for r in results])

        accuracies = np.array([r['accuracy'] for r in results])

        avg_acc = float(np.mean(accuracies))
        min_acc = float(np.min(accuracies))
        max_acc = float(np.max(accuracies))
        std_acc = float(np.std(accuracies))

        # weighted metrics (ignore invalid preds -1)
        valid_mask = all_y_pred >= 0
        if valid_mask.sum() == 0:
            precision = recall = f1 = 0.0
            support = None
            cm = None
        else:
            precision, recall, f1, support = precision_recall_fscore_support(
                all_y_val[valid_mask], all_y_pred[valid_mask], average='weighted', zero_division=0
            )
            cm = confusion_matrix(all_y_val[valid_mask], all_y_pred[valid_mask])

        model_summaries[model_name] = {
            'accuracy': avg_acc,
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1),
            'support': support,
            'confusion_matrix': cm,
            'min_accuracy': min_acc,
            'max_accuracy': max_acc,
            'std_accuracy': std_acc,
            'n_folds': len(results)
        }

    return all_results, model_summaries, le

# ---------------------------
# 5) Example usage (replace X_df, y_series with your data)
# ---------------------------
if __name__ == "__main__":
    X_df = X_balanced
    y_series = y_balanced
    all_results, model_summaries, label_encoder = run_all_models_with_kfold(
        X=X_df,
        y=y_series,
        n_splits=10,
        random_state=42,
        include_dnn=True,
        dnn_epochs=50,
        dnn_batch=32
    )

    print("\n\n============================================")
    print("FINAL MODEL SUMMARIES")
    print("============================================")

    for model_name, stats in model_summaries.items():
        print(f"\nModel: {model_name}")
        print(f"Mean Accuracy: {stats['accuracy']:.4f}")
        print(f"Min Accuracy:  {stats['min_accuracy']:.4f}")
        print(f"Max Accuracy:  {stats['max_accuracy']:.4f}")
        print(f"Std Dev:       {stats['std_accuracy']:.4f}")
        print(f"Precision:     {stats['precision']:.4f}")
        print(f"Recall:        {stats['recall']:.4f}")
        print(f"F1 Score:      {stats['f1']:.4f}")
        print("\nConfusion Matrix:")
        print(stats['confusion_matrix'])


=== Fold 1/10 ===
 Training SVM (Linear) ... acc=0.9038
 Training SVM (RBF Kernel) ... acc=0.9615
 Training SVM (Poly Kernel) ... acc=0.8269
 Training XGBoost ... acc=0.9808
 Training MLP ... acc=0.9231
 Training Decision Tree ... acc=0.8654
 Training Random Forest ... acc=0.9615
 Training CatBoost ... acc=0.9808
 Training GMM ... acc=0.5385
 Training LightGBM ... [LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001352 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12838
[LightGBM] [Info] Number of data points in the train set: 461, number of used features: 108
[LightGBM] [Info] Start training from score -1.096445
[LightGBM] [Info] Start training from score -1.096445
[LightGBM] [Info] Start training from score -1.102960
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

# **VOTING CODE**

In [42]:
# Sort models by accuracy descending
sorted_models = sorted(model_summaries.items(), key=lambda x: x[1]['accuracy'], reverse=True)

top3 = sorted_models[:3]

print("\n============================================")
print("TOP 3 MODELS SELECTED FOR VOTING ENSEMBLE")
print("============================================")
for name, stats in top3:
    print(f"{name}: Accuracy = {stats['accuracy']:.4f}")

def build_model_by_name(name):
    if name == "SVM (Linear)":
        return SVC(kernel='linear', probability=True, random_state=42)

    elif name == "SVM (RBF Kernel)":
        return SVC(kernel='rbf', probability=True, random_state=42, gamma='scale')

    elif name == "SVM (Poly Kernel)":
        return SVC(kernel='poly', degree=3, probability=True, random_state=42, gamma='scale')

    elif name == "XGBoost":
        return XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)

    elif name == "MLP":
        return MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500,
                             random_state=42, early_stopping=True)

    elif name == "Decision Tree":
        return DecisionTreeClassifier(random_state=42, max_depth=10)

    elif name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, random_state=42)

    elif name == "CatBoost":
        return CatBoostClassifier(verbose=0, random_state=42)

    elif name == "LightGBM":
        return LGBMClassifier(random_state=42)

    else:
        raise ValueError(f"Model not supported in ensemble: {name}")

def build_dnn_model(input_dim, num_classes):
    model = Sequential([
        Dense(256, activation='relu', input_dim=input_dim),
        BatchNormalization(),
        Dropout(0.4),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model



TOP 3 MODELS SELECTED FOR VOTING ENSEMBLE
XGBoost: Accuracy = 0.9552
Random Forest: Accuracy = 0.9552
DNN: Accuracy = 0.9532


In [44]:
from scipy.stats import mode

voting_results = []

X_array = X_df.values
y_array = y_series.values

# Build the list of top-3 model names
top3_names = [name for name, stats in top3]

n_classes = len(np.unique(y_array))
input_dim = X_array.shape[1]

for fold, (train_idx, val_idx) in enumerate(skf.split(X_array, y_array), 1):
    print(f"\nFold {fold}/{n_splits}...")

    X_train, X_val = X_array[train_idx], X_array[val_idx]
    y_train, y_val = y_array[train_idx], y_array[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    fold_preds = []

    # ============= Classical top-3 models =============
    for name in top3_names:
        if name != "DNN":  
            model = build_model_by_name(name)
            model.fit(X_train_scaled, y_train)
            pred = model.predict(X_val_scaled)
            fold_preds.append(pred)

    # ============= DNN model =============
    if "DNN" in top3_names:
        dnn = build_dnn_model(input_dim, n_classes)
        es = EarlyStopping(monitor='val_loss', patience=8,
                           restore_best_weights=True, verbose=0)
        dnn.fit(X_train_scaled, y_train,
                validation_data=(X_val_scaled, y_val),
                epochs=50, batch_size=32, verbose=0,
                callbacks=[es])
        prob = dnn.predict(X_val_scaled)
        pred = np.argmax(prob, axis=1)
        fold_preds.append(pred)

    # convert to array shape: (n_models, n_samples)
    fold_preds = np.array(fold_preds)

    # ============= Majority Voting =============
    final_pred = np.squeeze(mode(fold_preds, axis=0).mode)


    acc = accuracy_score(y_val, final_pred)
    print(f"Accuracy: {acc:.4f}")

    voting_results.append({
        'fold': fold,
        'accuracy': acc,
        'y_val': y_val,
        'y_pred': final_pred
    })
all_y_val = np.concatenate([r['y_val'] for r in voting_results])
all_y_pred = np.concatenate([r['y_pred'] for r in voting_results])
accuracies = np.array([r['accuracy'] for r in voting_results])

print("\nVoting Ensemble Stats:")
print(f"Mean Accuracy: {accuracies.mean():.4f}")
print(f"Min Accuracy : {accuracies.min():.4f}")
print(f"Max Accuracy : {accuracies.max():.4f}")
print(f"Std Dev      : {accuracies.std():.4f}")

precision, recall, f1, support = precision_recall_fscore_support(
    all_y_val, all_y_pred, average='weighted'
)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(all_y_val, all_y_pred))



Fold 1/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
Accuracy: 0.9808

Fold 2/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
Accuracy: 0.9808

Fold 3/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
Accuracy: 0.9038

Fold 4/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
Accuracy: 0.9608

Fold 5/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Accuracy: 0.9804

Fold 6/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Accuracy: 0.9216

Fold 7/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
Accuracy: 1.0000

Fold 8/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
Accuracy: 0.9412

Fold 9/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
Accuracy: 0.9412

Fold 10/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
Accuracy: 0.9608

Voting Ensemble Stats:
Mean Accuracy: 0.9571
Min Accuracy : 0.9038
Max Accuracy : 1.0000
Std Dev      : 0.0285
Precision: 0.9589
Recall   : 0.9571
F1-score : 0.9570

Confusion Matrix:
[[168   0   3]
 [  5 155  11]
 [  2   1 168]]


# **Final Model Summaries**

### Model: SVM (Linear)
- Mean Accuracy: 0.9103  
- Min Accuracy: 0.8627  
- Max Accuracy: 0.9608  
- Std Dev: 0.0292  
- Precision: 0.9141  
- Recall: 0.9103  
- F1 Score: 0.9096  

Confusion Matrix:  
- [165, 2, 4]  
- [11, 141, 19]  
- [7, 3, 161]  

---

### Model: SVM (RBF Kernel)
- Mean Accuracy: 0.9043  
- Min Accuracy: 0.8627  
- Max Accuracy: 0.9615  
- Std Dev: 0.0344  
- Precision: 0.9071  
- Recall: 0.9045  
- F1 Score: 0.9049  

Confusion Matrix:  
- [156, 2, 13]  
- [5, 151, 15]  
- [8, 6, 157]  

---

### Model: SVM (Poly Kernel)
- Mean Accuracy: 0.7913  
- Min Accuracy: 0.7255  
- Max Accuracy: 0.8846  
- Std Dev: 0.0433  
- Precision: 0.8521  
- Recall: 0.7914  
- F1 Score: 0.7940  

Confusion Matrix:  
- [170, 1, 0]  
- [44, 119, 8]  
- [53, 1, 117]  

---

### Model: XGBoost
- Mean Accuracy: 0.9552  
- Min Accuracy: 0.8654  
- Max Accuracy: 1.0000  
- Std Dev: 0.0387  
- Precision: 0.9561  
- Recall: 0.9552  
- F1 Score: 0.9551  

Confusion Matrix:  
- [164, 3, 4]  
- [4, 158, 9]  
- [2, 1, 168]  

---

### Model: MLP
- Mean Accuracy: 0.8634  
- Min Accuracy: 0.7843  
- Max Accuracy: 0.9615  
- Std Dev: 0.0648  
- Precision: 0.8641  
- Recall: 0.8635  
- F1 Score: 0.8638  

Confusion Matrix:  
- [151, 5, 15]  
- [2, 151, 18]  
- [19, 11, 141]  

---

### Model: Decision Tree
- Mean Accuracy: 0.8637  
- Min Accuracy: 0.8077  
- Max Accuracy: 0.9608  
- Std Dev: 0.0422  
- Precision: 0.8688  
- Recall: 0.8635  
- F1 Score: 0.8623  

Confusion Matrix:  
- [145, 11, 15]  
- [17, 133, 21]  
- [5, 1, 165]  

---

### Model: Random Forest
- Mean Accuracy: 0.9552  
- Min Accuracy: 0.9038  
- Max Accuracy: 1.0000  
- Std Dev: 0.0302  
- Precision: 0.9563  
- Recall: 0.9552  
- F1 Score: 0.9550  

Confusion Matrix:  
- [166, 2, 3]  
- [5, 156, 10]  
- [2, 1, 168]  

---

### Model: CatBoost
- Mean Accuracy: 0.9454  
- Min Accuracy: 0.8846  
- Max Accuracy: 0.9808  
- Std Dev: 0.0310  
- Precision: 0.9475  
- Recall: 0.9454  
- F1 Score: 0.9454  

Confusion Matrix:  
- [166, 2, 3]  
- [3, 153, 15]  
- [4, 1, 166]  

---

### Model: GMM
- Mean Accuracy: 0.4853  
- Min Accuracy: 0.3725  
- Max Accuracy: 0.5490  
- Std Dev: 0.0468  
- Precision: 0.4569  
- Recall: 0.4854  
- F1 Score: 0.4219  

Confusion Matrix:  
- [135, 30, 6]  
- [54, 103, 14]  
- [111, 49, 11]  

---

### Model: LightGBM
- Mean Accuracy: 0.9474  
- Min Accuracy: 0.8654  
- Max Accuracy: 0.9808  
- Std Dev: 0.0323  
- Precision: 0.9492  
- Recall: 0.9474  
- F1 Score: 0.9473  

Confusion Matrix:  
- [166, 2, 3]  
- [3, 154, 14]  
- [4, 1, 166]  

---

### Model: DNN
- Mean Accuracy: 0.9532  
- Min Accuracy: 0.9216  
- Max Accuracy: 1.0000  
- Std Dev: 0.0234  
- Precision: 0.9550  
- Recall: 0.9532  
- F1 Score: 0.9531  

Confusion Matrix:  
- [169, 1, 1]  
- [1, 154, 16]  
- [3, 2, 166]  


# **VOTING ENSEMBLE!**
## Voting Ensemble Stats:
- Mean Accuracy: 0.9571
- Min Accuracy : 0.9038
- Max Accuracy : 1.0000
- Std Dev      : 0.0285
- Precision: 0.9589
- Recall   : 0.9571
- F1-score : 0.9570
### Confusion Matrix:
[[168   0   3]
 [  5 155  11]
 [  2   1 168]]

# **RHYTHM DATASET ML TRAINING**

In [45]:
X = df_rhythm.drop('class_encoded', axis=1)
y = df_rhythm['class_encoded']

In [46]:
X_balanced, y_balanced = balance_dataset(X, y)
print("\nBalanced class distribution:")
print(y_balanced.value_counts())



Balancing to 171 samples per class

Balanced class distribution:
class_encoded
2    171
0    171
1    171
Name: count, dtype: int64


In [47]:
if __name__ == "__main__":
    X_df = X_balanced
    y_series = y_balanced
    all_results, model_summaries, label_encoder = run_all_models_with_kfold(
        X=X_df,
        y=y_series,
        n_splits=10,
        random_state=42,
        include_dnn=True,
        dnn_epochs=50,
        dnn_batch=32
    )

    print("\n\n============================================")
    print("FINAL MODEL SUMMARIES RHYTHM DATA")
    print("============================================")

    for model_name, stats in model_summaries.items():
        print(f"\nModel: {model_name}")
        print(f"Mean Accuracy: {stats['accuracy']:.4f}")
        print(f"Min Accuracy:  {stats['min_accuracy']:.4f}")
        print(f"Max Accuracy:  {stats['max_accuracy']:.4f}")
        print(f"Std Dev:       {stats['std_accuracy']:.4f}")
        print(f"Precision:     {stats['precision']:.4f}")
        print(f"Recall:        {stats['recall']:.4f}")
        print(f"F1 Score:      {stats['f1']:.4f}")
        print("\nConfusion Matrix:")
        print(stats['confusion_matrix'])


=== Fold 1/10 ===
 Training SVM (Linear) ... acc=0.6923
 Training SVM (RBF Kernel) ... acc=0.6731
 Training SVM (Poly Kernel) ... acc=0.6154
 Training XGBoost ... acc=0.8846
 Training MLP ... acc=0.6346
 Training Decision Tree ... acc=0.7885
 Training Random Forest ... acc=0.8846
 Training CatBoost ... acc=0.8654
 Training GMM ... acc=0.5000
 Training LightGBM ... [LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000129 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1186
[LightGBM] [Info] Number of data points in the train set: 461, number of used features: 9
[LightGBM] [Info] Start training from score -1.096445
[LightGBM] [Info] Start training from score -1.096445
[LightGBM] [Info] Start training from score -1.102960
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

In [49]:
from scipy.stats import mode
top_3 = sorted(model_summaries.items(), key=lambda x: x[1]['accuracy'], reverse=True)[:3]
voting_results = []

X_array = X_df.values
y_array = y_series.values

# Build the list of top-3 model names
top3_names = [name for name, stats in top3]

n_classes = len(np.unique(y_array))
input_dim = X_array.shape[1]

for fold, (train_idx, val_idx) in enumerate(skf.split(X_array, y_array), 1):
    print(f"\nFold {fold}/{n_splits}...")

    X_train, X_val = X_array[train_idx], X_array[val_idx]
    y_train, y_val = y_array[train_idx], y_array[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    fold_preds = []

    # ============= Classical top-3 models =============
    for name in top3_names:
        if name != "DNN":  
            model = build_model_by_name(name)
            model.fit(X_train_scaled, y_train)
            pred = model.predict(X_val_scaled)
            fold_preds.append(pred)

    # ============= DNN model =============
    if "DNN" in top3_names:
        dnn = build_dnn_model(input_dim, n_classes)
        es = EarlyStopping(monitor='val_loss', patience=8,
                           restore_best_weights=True, verbose=0)
        dnn.fit(X_train_scaled, y_train,
                validation_data=(X_val_scaled, y_val),
                epochs=50, batch_size=32, verbose=0,
                callbacks=[es])
        prob = dnn.predict(X_val_scaled)
        pred = np.argmax(prob, axis=1)
        fold_preds.append(pred)

    # convert to array shape: (n_models, n_samples)
    fold_preds = np.array(fold_preds)

    # ============= Majority Voting =============
    final_pred = np.squeeze(mode(fold_preds, axis=0).mode)


    acc = accuracy_score(y_val, final_pred)
    print(f"Accuracy: {acc:.4f}")

    voting_results.append({
        'fold': fold,
        'accuracy': acc,
        'y_val': y_val,
        'y_pred': final_pred
    })
all_y_val = np.concatenate([r['y_val'] for r in voting_results])
all_y_pred = np.concatenate([r['y_pred'] for r in voting_results])
accuracies = np.array([r['accuracy'] for r in voting_results])

print("\nVoting Ensemble Stats:")
print(f"Mean Accuracy: {accuracies.mean():.4f}")
print(f"Min Accuracy : {accuracies.min():.4f}")
print(f"Max Accuracy : {accuracies.max():.4f}")
print(f"Std Dev      : {accuracies.std():.4f}")

precision, recall, f1, support = precision_recall_fscore_support(
    all_y_val, all_y_pred, average='weighted'
)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(all_y_val, all_y_pred))



Fold 1/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Accuracy: 0.8846

Fold 2/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
Accuracy: 0.9423

Fold 3/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Accuracy: 0.7308

Fold 4/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Accuracy: 0.8824

Fold 5/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Accuracy: 0.8039

Fold 6/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
Accuracy: 0.8824

Fold 7/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Accuracy: 0.8627

Fold 8/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
Accuracy: 0.9020

Fold 9/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
Accuracy: 0.9020

Fold 10/10...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Accuracy: 0.9412

Voting Ensemble Stats:
Mean Accuracy: 0.8734
Min Accuracy : 0.7308
Max Accuracy : 0.9423
Std Dev      : 0.0605
Precision: 0.8758
Recall   : 0.8733
F1-score : 0.8732

Confusion Matrix:
[[145  10  16]
 [ 12 144  15]
 [  9   3 159]]


# Rhythm data results:
Voting Ensemble Stats:
- Mean Accuracy: 0.8734
- Min Accuracy : 0.7308
- Max Accuracy : 0.9423
- Std Dev      : 0.0605
- Precision: 0.8758
- Recall   : 0.8733
- F1-score : 0.8732

Confusion Matrix:
- [[145  10  16]
-  [ 12 144  15]
-  [  9   3 159]]

# Rhythm Data Results:
============================================
FINAL MODEL SUMMARIES RHYTHM DATA
============================================

### Model: SVM (Linear)
- Mean Accuracy: 0.6275  
- Min Accuracy: 0.5294  
- Max Accuracy: 0.7308  
- Std Dev: 0.0612  
- Precision: 0.6208  
- Recall: 0.6277  
- F1 Score: 0.6144  

Confusion Matrix:
[[130  14  27]
 [ 16 130  25]
 [ 75  34  62]]

### Model: SVM (RBF Kernel)
- Mean Accuracy: 0.6471  
- Min Accuracy: 0.5686  
- Max Accuracy: 0.7647  
- Std Dev: 0.0595  
- Precision: 0.6487  
- Recall: 0.6472  
- F1 Score: 0.6354  

Confusion Matrix:
[[129  14  28]
 [ 17 138  16]
 [ 82  24  65]]

### Model: SVM (Poly Kernel)
- Mean Accuracy: 0.5495  
- Min Accuracy: 0.4706  
- Max Accuracy: 0.6346  
- Std Dev: 0.0591  
- Precision: 0.6215  
- Recall: 0.5497  
- F1 Score: 0.5271  

Confusion Matrix:
[[151   3  17]
 [ 60  94  17]
 [126   8  37]]

### Model: XGBoost
- Mean Accuracy: 0.8598  
- Min Accuracy: 0.7308  
- Max Accuracy: 0.9216  
- Std Dev: 0.0521  
- Precision: 0.8636  
- Recall: 0.8596  
- F1 Score: 0.8596  

Confusion Matrix:
[[138  11  22]
 [  9 146  16]
 [ 10   4 157]]

### Model: MLP
- Mean Accuracy: 0.6063  
- Min Accuracy: 0.5000  
- Max Accuracy: 0.6667  
- Std Dev: 0.0464  
- Precision: 0.5870  
- Recall: 0.6062  
- F1 Score: 0.5871  

Confusion Matrix:
[[119  19  33]
 [  7 140  24]
 [ 79  40  52]]

### Model: Decision Tree
- Mean Accuracy: 0.7917  
- Min Accuracy: 0.6863  
- Max Accuracy: 0.9216  
- Std Dev: 0.0717  
- Precision: 0.7954  
- Recall: 0.7914  
- F1 Score: 0.7916  

Confusion Matrix:
[[137  16  18]
 [ 14 128  29]
 [ 24   6 141]]

### Model: Random Forest
- Mean Accuracy: 0.8695  
- Min Accuracy: 0.7500  
- Max Accuracy: 0.9423  
- Std Dev: 0.0546  
- Precision: 0.8721  
- Recall: 0.8694  
- F1 Score: 0.8692  

Confusion Matrix:
[[141  13  17]
 [  9 146  16]
 [  9   3 159]]

### Model: CatBoost
- Mean Accuracy: 0.8773  
- Min Accuracy: 0.7885  
- Max Accuracy: 0.9038  
- Std Dev: 0.0363  
- Precision: 0.8802  
- Recall: 0.8772  
- F1 Score: 0.8769  

Confusion Matrix:
[[141  15  15]
 [  5 149  17]
 [  9   2 160]]

### Model: GMM
- Mean Accuracy: 0.4990  
- Min Accuracy: 0.4423  
- Max Accuracy: 0.6078  
- Std Dev: 0.0547  
- Precision: 0.5505  
- Recall: 0.4990  
- F1 Score: 0.4446  

Confusion Matrix:
[[143  18  10]
 [ 73  96   2]
 [117  37  17]]

### Model: LightGBM
- Mean Accuracy: 0.8618  
- Min Accuracy: 0.7500  
- Max Accuracy: 0.9216  
- Std Dev: 0.0440  
- Precision: 0.8642  
- Recall: 0.8616  
- F1 Score: 0.8616  

Confusion Matrix:
[[140  10  21]
 [ 10 147  14]
 [ 10   6 155]]

### Model: DNN
- Mean Accuracy: 0.6647  
- Min Accuracy: 0.5577  
- Max Accuracy: 0.7692  
- Std Dev: 0.0688  
- Precision: 0.6707  
- Recall: 0.6647  
- F1 Score: 0.6609  

Confusion Matrix:
[[129   8  34]
 [ 17 134  20]
 [ 75  18  78]]
